In [4]:
import pandas as pd
import numpy as np

In [5]:
fake_df = pd.read_csv(r"C:\Users\anish\OneDrive\Desktop\Fake_News_Detection\data\fake.csv")
true_df = pd.read_csv(r"C:\Users\anish\OneDrive\Desktop\Fake_News_Detection\data\true.csv")

print("Fake shape:", fake_df.shape)
print("True shape:", true_df.shape)

Fake shape: (23481, 4)
True shape: (21417, 4)


In [6]:
fake_df.isnull().sum()


title      0
text       0
subject    0
date       0
dtype: int64

In [7]:
fake_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    23481 non-null  str  
 1   text     23481 non-null  str  
 2   subject  23481 non-null  str  
 3   date     23481 non-null  str  
dtypes: str(4)
memory usage: 733.9 KB


In [8]:
true_df.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [9]:
true_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    21417 non-null  str  
 1   text     21417 non-null  str  
 2   subject  21417 non-null  str  
 3   date     21417 non-null  str  
dtypes: str(4)
memory usage: 669.4 KB


In [10]:
fake_df["label"] = 0   # Fake = 0
true_df["label"] = 1   # Real = 1

In [11]:
df = pd.concat([fake_df, true_df], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.shape

(44898, 5)

In [12]:
df["label"].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

In [13]:
df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]

df.head()

,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1


In [14]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["content"] = df["content"].apply(clean_text)

df.head()

,content,label
0,ben stein calls out th circuit court committed...,0
1,trump drops steve bannon from national securit...,1
2,puerto rico expects us to lift jones act shipp...,1
3,oops trump just accidentally confirmed he leak...,0
4,donald trump heads for scotland to reopen a go...,1


In [15]:
df.shape

(44898, 2)

In [25]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (35918,)
Testing size: (8980,)


In [27]:
import os

os.makedirs("model", exist_ok=True)

joblib.dump(X_test, "model/X_test.pkl")
joblib.dump(y_test, "model/y_test.pkl")

['model/y_test.pkl']

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,     # limit features (good for beginner project)
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (35918, 5000)
Testing TF-IDF shape: (8980, 5000)


In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model training completed.")

Model training completed.


In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9864142538975501

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4710
           1       0.98      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [20]:
import pickle
import os

# create model folder if not exists
os.makedirs("../model", exist_ok=True)

# save model
with open("../model/model.pkl", "wb") as f:
    pickle.dump(model, f)

# save vectorizer
with open("../model/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully.")

Model and vectorizer saved successfully.


In [24]:
print(df['label'].value_counts())

label
0    23481
1    21417
Name: count, dtype: int64
